# Fase 4 — Ajuste do T5 para gerar manchetes em português

**Projeto:** fiap-fase4-manchetes
**Ambiente:** Google Colab, GPU Tesla T4
**Modelo base:** `unicamp-dl/ptt5-base-portuguese-vocab` (T5 220M, pré-treinado em português)

Este notebook:
1. confere a GPU;
2. clona o repositório do projeto;
3. ajusta o modelo nos dados de treino;
4. avalia o modelo **base** e o **ajustado** no mesmo conjunto de teste;
5. publica o modelo no Hugging Face Hub (opcional) para o Streamlit consumir.

> Antes de rodar: **Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU**.

## 1. Conferir a GPU

In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("cuda disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("memória:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## 2. Clonar o repositório

Troque `USUARIO` pelo seu usuário do GitHub. Se o repositório ainda não estiver
publicado, faça upload dos arquivos do projeto para o Colab (ícone de pasta) e
pule esta célula; mantenha o diretório de trabalho na raiz do projeto.

In [ ]:
USUARIO = "SEU_USUARIO"  # <-- ajuste aqui
REPO = f"https://github.com/{USUARIO}/fiap-fase4-manchetes.git"

import os
if not os.path.isdir("/content/fiap-fase4-manchetes"):
    !git clone {REPO} /content/fiap-fase4-manchetes
%cd /content/fiap-fase4-manchetes
!ls

## 3. Instalar dependências

In [ ]:
!pip install -q -r requirements-train.txt
import transformers, torch
print("transformers:", transformers.__version__, "| torch:", torch.__version__)

## 4. Conferir os dados

Os conjuntos já versionados em `data/splits/` foram divididos **por acontecimento**,
de modo que notícias do mesmo evento não aparecem em treino e teste ao mesmo tempo.

In [ ]:
import json, pandas as pd
from pathlib import Path

print(json.dumps(json.loads(Path("data/splits/resumo.json").read_text()), indent=2, ensure_ascii=False))
for nome in ("treino", "validacao", "teste"):
    df = pd.read_csv(f"data/splits/{nome}.csv")
    print(f"{nome:10} {len(df):5} notícias | grupos: {df['grupo'].nunique():5}")
df.head(3)[["titulo", "entrada"]]

## 5. Ajustar o modelo

`--fp16` é obrigatório na T4 para caber em memória e acelerar.
O treinamento dura cerca de 15 a 25 minutos (6 épocas, 2400 exemplos).

Se quiser publicar direto no Hub, crie um token em
<https://huggingface.co/settings/tokens> e cadastre como *secret* `HF_TOKEN`
(ícone de chave na barra lateral). Depois informe `--push-hub SEU_USUARIO/manchetes-ptt5`.

In [ ]:
!python scripts/treinar_modelo.py \
    --modelo unicamp-dl/ptt5-base-portuguese-vocab \
    --dados data/splits \
    --saida modelos/manchetes-ptt5 \
    --epocas 6 --batch 8 --grad-accum 2 --lr 2e-4 \
    --fp16

## 6. Resultados do treinamento

In [ ]:
import json
from pathlib import Path

rel = json.loads(Path("modelos/manchetes-ptt5/metricas_treino.json").read_text())
print("Treino:", rel["treino"])
print("Validação:", {k: round(v, 4) for k, v in rel["validacao"].items() if isinstance(v, float)})
print("Ambiente:", rel["ambiente"])
for exemplo in rel["exemplos_gerados"]:
    print("\nREFERÊNCIA:", exemplo["titulo_referencia"])
    for i, s in enumerate(exemplo["sugestoes"], 1):
        print(f"  {i}. {s}")

## 7. Avaliar modelo base e modelo ajustado no teste

Os dois comandos usam o mesmo conjunto de teste, o mesmo prefixo e a mesma
decodificação (3 sugestões, 8 feixes). Isso torna a comparação justa.

O conjunto de teste tem 300 notícias e a avaliação completa leva poucos minutos na T4.
Se o tempo apertar, use `--limite 100` **nos dois** comandos para manter a comparação justa.

In [ ]:
!python scripts/avaliar_modelo.py \
    --modelo unicamp-dl/ptt5-base-portuguese-vocab \
    --rotulo base

In [ ]:
!python scripts/avaliar_modelo.py \
    --modelo modelos/manchetes-ptt5 \
    --rotulo ajustado

In [ ]:
!python scripts/avaliar_modelo.py --comparar avaliacao/base.json avaliacao/ajustado.json

## 8. Ver exemplos lado a lado

Use esta saída para o vídeo e para registrar falhas qualitativas.

In [ ]:
import pandas as pd

base = pd.read_csv("avaliacao/base_previsoes.csv")
ajustado = pd.read_csv("avaliacao/ajustado_previsoes.csv")
junto = base.merge(ajustado, on=["id", "titulo_referencia"], suffixes=("_base", "_ajustado"))
for linha in junto.head(8).itertuples(index=False):
    print("REFERÊNCIA:", linha.titulo_referencia)
    print("  base    :", linha.sugestao_1_base)
    print("  ajustado:", linha.sugestao_1_ajustado)
    print()

## 9. Publicar o modelo no Hugging Face Hub

Necessário para o Streamlit carregar o modelo ajustado. Cadastre o token como
*secret* `HF_TOKEN` antes de rodar.

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi

token = userdata.get("HF_TOKEN")  # se falhar, use: token = "hf_..."
REPO_ID = "SEU_USUARIO/manchetes-ptt5"  # <-- ajuste aqui

api = HfApi(token=token)
api.create_repo(REPO_ID, exist_ok=True)
api.upload_folder(
    folder_path="modelos/manchetes-ptt5",
    repo_id=REPO_ID,
    ignore_patterns=["checkpoints/*"],
)
api.upload_file(
    path_or_fileobj="avaliacao/comparacao.json",
    path_in_repo="comparacao.json",
    repo_id=REPO_ID,
)
print("Publicado em https://huggingface.co/" + REPO_ID)

## 10. Baixar resultados para o repositório local

Baixe `avaliacao/` e `metricas_treino.json` e copie para o projeto local.
O modelo em si não precisa ser baixado: o Streamlit lê do Hub.

In [ ]:
!zip -qr resultados.zip avaliacao modelos/manchetes-ptt5/metricas_treino.json
from google.colab import files
files.download("resultados.zip")